# Loss functions



## Multimoal Loss

GMM like multimodal outputs k different modes, while the ground truth only has one mode. The naive GMM loss is to minmize the negative log likelihood of the ground truth under the GMM distribution. However, this loss is no ideal as it leads to mode collapse, where the model only learns one mode and ignores the others. 

To address this issue, the winner-takes-all loss is proposed, which only considers the mode that is closest to the ground truth. This loss encourages the model to learn multiple modes and prevents mode collapse.

The loss can be formulated as the sum of two parts: one is a classification loss as cross-entropy that encourages the model to assign high probability to the mode that is closest to the ground truth, and the other is a regression loss that encourages the model to predict the ground truth accurately for that mode.



## Weighed Cross Entropy Loss
In some cases, the data may be imbalanced, meaning that some classes are more frequent than others. In such cases, the standard cross-entropy loss may not perform well, as it may be biased towards the majority class. To address this issue, a weighted cross-entropy loss can be used, where different weights are assigned to different classes based on their frequency in the data. This way, the model can learn to pay more attention to the minority classes and improve its performance on those classes.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedCrossEntropyLoss(nn.Module):
    def __init__(self, class_weights):
        super().__init__()

        # tensor of shape [num_classes]
        self.register_buffer(
            "class_weights",
            torch.tensor(class_weights, dtype=torch.float32)
        )

    def forward(self, logits, targets):
        """
        logits: [B, C]
        targets: [B]
        """

        loss = F.cross_entropy(
            logits,
            targets,
            weight=self.class_weights
        )

        return loss

## Focal Loss
Focal loss is a modification of the cross-entropy loss that is designed to address the issue of class imbalance in object detection tasks. It works by down-weighting the loss assigned to well-classified examples, which are typically the majority class, and up-weighting the loss assigned to misclassified examples, which are typically the minority class. This way, the model can focus more on learning from the hard examples and improve its performance on the minority class. The focal loss can be formulated as:

$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

where $p_t$ is the predicted probability of the true class, $\alpha_t$ is a weighting factor for the class, and $\gamma$ is a focusing parameter that controls the down-weighting of well-classified examples. When $\gamma=0$, the focal loss reduces to the standard cross-entropy loss. As $\gamma$ increases, the loss assigned to well-classified examples decreases, and the model focuses more on learning from the hard examples.

cross entropy loss:

$$ce = \sum_i^k  y_i \log p_i =  -\log(p_t)$$

Because $y_i$ is one-hot encoding, only the term corresponding to the true class contributes to the loss, which is $-\log(p_t)$ where $p_t$ is the predicted probability of the true class. The focal loss modifies this by adding a weighting factor $\alpha_t$ and a focusing parameter $\gamma$ to down-weight well-classified examples and up-weight misclassified examples, as shown in the formula above.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        """
        logits: [B, C]
        targets: [B]
        """

        ce_loss = F.cross_entropy(
            logits,
            targets,
            reduction='none'
        )

        pt = torch.exp(-ce_loss)

        focal_loss = (
            self.alpha *
            (1 - pt) ** self.gamma *
            ce_loss
        )

        return focal_loss.mean()